## Fase 2: Exploración Inicial del Dataset, Limpieza y Transformación

### Objetivo

Realizar la obtención y exploración inicial del conjunto de datos de startups para identificar su estructura, calidad y características principales.

In [ ]:
#verificar versiones
import pandas as pd
import numpy as np
import sys

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

In [ ]:
#Ubicación

from pathlib import Path

def localizar_raiz():
    carpeta_actual = Path.cwd()

    if carpeta_actual.name in {"F1", "F2", "F3", "F4"}:
        return carpeta_actual.parent

    return carpeta_actual

RAIZ = localizar_raiz()

print("Raíz del proyecto:")
print(RAIZ)

## Archivo a DataFrame

Cargamos el notebook que está en nuestro repositorio GitHub y se crea el DataFrame para proceder con exploración, limpieza y transformación de los datos

In [ ]:
#Creación de data Frame, carga de CSV desde Repositorio Github
RUTA_DATASET = "../data/y-combinator-companies.csv"

df_original = pd.read_csv(RUTA_DATASET)
#copia de dataframe para no sobreescribir el original
df = df_original.copy()
df.head()

In [ ]:
df.describe()

### Exploración DataFrame

El primer paso es revisar la estructura del conjunto de datos para identificar las variables necesarias que resolverán nuestro problema identificado en la Fase 1

In [ ]:
# Revisar estructura del dataset
def exploracion(df):
    print("Colmunas:", df.shape[0])
    print("Filas:", df.shape[1])
    print("Duplicados:",df.duplicated().sum())
    print("Valores no válidos por Variable:", df.isnull().sum())
    print("Tipos de datos:", df.dtypes)
exploracion(df)

In [ ]:
# Revisar estadísticas descriptivas
df.describe()

In [ ]:
# Conteo de categorias en las variables nominales más importantes
variables_consideradas = ['is_operating', 'is_hiring', 'nonprofit',
                        'industry', 'status','is_ai','hq_country']
for variables in variables_consideradas:
    print(f"\n{variables}:")
    #Identificar faltantes
    print(df[variables].value_counts(dropna=False))

In [ ]:
# Diccionario de variables consideradas, se seleccionan las más importantes
# observado en el archivo: no se afirma nada que no se verifique.
CONJUNTO_DATOS = [
    ("is_operating",     "binaria",              "¿Está Operativa?: True = sí, False = no"),
    ("is_hiring",        "binaria",              "¿Está contratando?: True = sí, False = no"),
    ("nonprofit",        "binaria",              "¿Sin Fines de Lucro?: True = sí, False = no "),
    ("status",           "nominal",              "Estado de la empresa:  Active, Inactive, Acquired o Public"),
    ("is_ai",            "binaria",              "¿Relacionada a IA?: True = sí, False = no"),
    ("team_size",        "cuantitativa discreta","Tamaño del equipo de la empresa"),
    ("industry",         "nominal",              "Industria a la que pertenece la empresa"),
    ("hq_country",       "nominal",              "País donde se encuentra la casa matriz de la empresa"),
    ("years_since_batch","cuantitativa discreta","Años transcurridos desde que la empresa participó en YC")
 
]
conjunto = pd.DataFrame(CONJUNTO_DATOS, columns=["variable", "rol_analitico", "descripcion"])

# Variables y tipos de datos
conjunto["dtype"] = [str(df_original[v].dtype) for v in conjunto["variable"]]
# nunique() cuenta valores distintos: distingue una binaria de una continua
conjunto["n_unicos"] = [int(df_original[v].nunique()) for v in conjunto["variable"]]
# Sobre booleanos, mean() devuelve la PROPORCIÓN de True: por 100 da el porcentaje
conjunto["pct_nulos"] = [round(df_original[v].isna().mean() * 100, 1)
                            for v in conjunto["variable"]]

# El diccionario es un entregable, no una tabla de paso: se guarda. Reemplaza al de la
# Fase 1 a propósito: conserva sus columnas y le agrega lo observado en el archivo.
conjunto.to_csv(DIR_DOCS / "diccionario_variables.csv", index=False)
conjunto

### Limpieza y selección de Datos
Nuestro dataframe contiene muchas columnas, por lo tanto, para no eliminar una cantidad grande de columnas se procede a seleccionar las variables de interés que nos ayudarán a resolver el problema

In [ ]:
variables_analisis = ["is_operating", "is_hiring","nonprofit",
                     "status","is_ai","team_size","industry",
                      "hq_country","years_since_batch","name"]
df = df_original[variables_analisis].copy()
df.head()

### Identificación y Reemplazo de variables nulos
En el siguiente código vemos los nulos de las variables seleccionadas, esos valores serán reemplazados por la mediana y si no es uan variable numérica se escribirá "No Identificado"

In [ ]:
df.isnull().sum()

In [ ]:
# Buscar registros con valores faltantes o negativos
df[
    df["years_since_batch"].isna() |
    (df["years_since_batch"] < 0)
]

In [ ]:
#La variable years_since_batch presentó un valor negativo y un valor faltante.
#El valor negativo se cambia por cero
#se elimina la variable con dato vacio, years_since_batch, debido a que corresponde a la misma empresa Ycombinator
# Se elimina para que no afecte al cálculo de los demás registros

def limpiar_datos(df):

    # Eliminar registro de Y Combinator
    filas_antes = len(df)

    df = df[df["name"] != "Y Combinator"]

    filas_eliminadas = filas_antes - len(df)

    print(
        f"Se eliminaron {filas_eliminadas} registro correspondiente a Y Combinator"
    )

    # team_size,en tamaño de equipo se reemplazan nulos por la mediana
    nulos_team = df["team_size"].isna().sum()

    mediana_team = df["team_size"].median()

    df["team_size"] = df["team_size"].fillna(mediana_team)

    print(
        f"team_size: {nulos_team} nulos reemplazados por la mediana ({mediana_team})"
    )

    # hq_country,a país de casa matriz se reemplazan nulos por desconocido
    nulos_pais = df["hq_country"].isna().sum()

    df["hq_country"] = df["hq_country"].fillna("Desconocido")

    print(
        f"hq_country: {nulos_pais} nulos reemplazados por 'Desconocido'"
    )

    # years_since_batch, a años desde participación de programa se reemplaza negativo por cero
    negativos = (df["years_since_batch"] < 0).sum()

    df.loc[
        df["years_since_batch"] < 0,
        "years_since_batch"
    ] = 0

    print(
        f"years_since_batch: {negativos} valores negativos reemplazados por 0"
    )

    return df

In [ ]:
#ejecutar limpieza de nulos en dataframe
df = limpiar_nulos(df)
print(df.isna().sum())

## Validación

Se confirma que el dataset tenga column as limpias de valores nulos o vacíos, y considere las variables más relevantes

In [ ]:
print("Dimensiones finales:")
print(df.shape)

print("\nValores nulos:")
print(df.isna().sum())

print("\nTipos de datos:")
print(df.dtypes)

## Trazabilidad

Se mantuvo el conjunto de datos original en el DataFrame df_original, realizando todas las transformaciones sobre una copia de trabajo (df).

Las acciones realizadas fueron:

1) Selección de variables relevantes.
2) Imputación de valores faltantes en team_size.
3) Reemplazo de valores faltantes en hq_country.
4) Corrección de valores negativos en years_since_batch.
5) Eliminación del registro correspondiente a Y Combinator.

### Visualización

Gráfico para visualizar la cantidad de startups por industria

In [ ]:
import matplotlib.pyplot as plt

df["industry"].value_counts().plot(
    kind="bar",
    figsize=(10,5)
)

plt.title("Cantidad de startups por industria")
plt.xlabel("Industria")
plt.ylabel("Cantidad")
plt.xticks(rotation=45)

plt.show()

## Conclusiones
Durante la exploración se identificaron las variables relevantes para el análisis de las empresas de Y Combinator.

Se detectaron valores faltantes en las variables team_size, hq_country y years_since_batch.

Para asegurar la calidad de los datos:

1) Los valores faltantes de team_size fueron reemplazados por la mediana.
2) Los valores faltantes de hq_country fueron reemplazados por la categoría "Desconocido".
3) El valor negativo detectado en years_since_batch fue reemplazado por 0.
4) Se eliminó el registro correspondiente a Y Combinator por no representar una startup.
5) Finalmente, el conjunto de datos quedó sin valores faltantes en las variables analizadas y preparado para etapas posteriores del proyecto.

## Reflexión Técnica

### Hallazgos

La mayoría de las empresas del dataset continúan operando, por lo que existe una diferencia importante entre empresas activas y no activas.

La variable team_size presenta una alta dispersión. Mientras la mediana es 6, existen empresas con miles de trabajadores, por lo que se utilizó la mediana para imputar los valores faltantes.

La variable hq_country presentó algunos registros sin información de país, los cuales fueron reemplazados por la categoría "Desconocido".

Dificultades y soluciones

Se identificó un registro correspondiente a Y Combinator que no representaba una startup, por lo que fue eliminado.
Se detectó un valor negativo en years_since_batch, el cual fue corregido para mantener la consistencia de los datos.
Los valores faltantes fueron tratados según el tipo de variable, utilizando mediana para datos numéricos y categorías descriptivas para datos categóricos.


### Aspectos a revisar en fases posteriores

1) Analizar si la antigüedad de las empresas influye en su estado operativo.
2) Evaluar la relación entre inteligencia artificial, industria y contratación.
3) Estudiar con mayor detalle las empresas con tamaños de equipo excepcionalmente altos.
